In [2]:
import os
import glob
from pypdf import PdfReader
import numpy as np
import tiktoken
from openai import OpenAI
from typing import List, Dict, Tuple
from dotenv import load_dotenv

# Read the OpenAI API key from the environment. This keeps the key private and makes the notebook easy to run on another machine.
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Fail early with a clear error message if the key isn’t set.
if not OPENAI_API_KEY:
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

client = OpenAI(api_key=OPENAI_API_KEY)

DOCS_DIR = os.path.expanduser("~/Desktop/RAG/docs")

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

# Chunk size in tokens. Smaller chunks retrieve more precisely; bigger chunks keep more context.
CHUNK_TOKENS = 400

# Small overlap so we don’t cut important info at chunk boundaries.

CHUNK_OVERLAP = 40
TOP_K = 5
MAX_CONTEXT_MESSAGES = 10

## Each token represents a meaning. The bigger the chunk, the more context it has to carry.
# Similarly, smaller tokens are more time consuming to analyze.

# Create the OpenAI client once and reuse it for embeddings + answers.
client = OpenAI(api_key=OPENAI_API_KEY)


In [3]:
def load_pdf_text(path: str) -> str:
    """
    Extract text from a PDF into one big string.
    Skips pages that return None from extract_text().
    """
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        txt = page.extract_text()
        if txt:
            pages.append(txt)
    return "\n\n".join(pages)

def load_all_texts(docs_dir: str):
    """
    Read .txt / .md / .pdf from docs_dir.
    Returns list of {"source": filename, "text": full_text}
    """
    texts = []

    # 1) .txt and .md
    patterns = ["*.txt", "*.md"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(docs_dir, p)))

    # 2) .pdf
    files.extend(glob.glob(os.path.join(docs_dir, "*.pdf")))

    for path in files:
        try:
            ext = os.path.splitext(path)[1].lower()
            if ext in [".txt", ".md"]:
                with open(path, "r", encoding="utf-8") as f:
                    full_text = f.read()
            elif ext == ".pdf":
                full_text = load_pdf_text(path)
            else:
                continue

            texts.append({
                "source": os.path.basename(path),
                "text": full_text
            })
        except Exception as e:
            print(f"Skipping {path}: {e}")

    return texts


def chunk_text(text: str,
               source: str,
               enc,
               chunk_tokens: int = CHUNK_TOKENS,
               overlap_tokens: int = CHUNK_OVERLAP):
    """Split a long document into smaller token-based chunks (with overlap) for embedding + retrieval."""
    tokens = enc.encode(text)
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        # Keep making chunks until we reach the end of the token list.
        end = start + chunk_tokens
        piece_tokens = tokens[start:end]
        piece_text = enc.decode(piece_tokens)
        chunks.append({
            "source": source,
            "chunk_id": idx,
            "text": piece_text
        })
        idx += 1
        start = end - overlap_tokens
        if start < 0:
            start = 0
    return chunks


def build_corpus(docs_dir: str):
    # Load all docs from a folder and split them into token chunks (this becomes our retrieval corpus).
    enc = tiktoken.get_encoding("cl100k_base")
    raw_docs = load_all_texts(docs_dir)
    all_chunks = []
    for d in raw_docs:
        pieces = chunk_text(d["text"], d["source"], enc)
        all_chunks.extend(pieces)
    return all_chunks

DOCS_DIR = os.path.expanduser("~/Desktop/RAG/docs")
chunks = build_corpus(DOCS_DIR)
print("Total chunks:", len(chunks))
print("Example chunk preview:\n", chunks[0]["text"][:500])


Total chunks: 1112
Example chunk preview:
 Aurélien Géron
Hands-on  
Machine Learning  
 with Scikit-Learn,  
Keras & TensorFlow
Concepts, Tools, and Techniques  
to Build Intelligent Systems
TM
2nd Edition
Updated for  TensorFlow 2


Aurélien Géron
Hands-On Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing

978-1-492-03264-9
[TI]
Hands-On Machine Learning with Scikit-Learn, Ker


In [4]:
[
  {"source": "book.txt", "chunk_id": 0, "text": "..."},
  {"source": "book.txt", "chunk_id": 1, "text": "..."},
  ...
]


[{'source': 'book.txt', 'chunk_id': 0, 'text': '...'},
 {'source': 'book.txt', 'chunk_id': 1, 'text': '...'},
 Ellipsis]

In [5]:


# make sure you still have OPENAI_API_KEY and client defined in the notebook
# if not, re-run the cell where you set OPENAI_API_KEY and did:
# client = OpenAI(api_key=OPENAI_API_KEY)

EMBED_MODEL = "text-embedding-3-small"

# The embeddings API has a token limit per request. If we send too many chunks at once, the total tokens can exceed 8192.
# Instead of batching by “64 items”, we batch by token budget so indexing never randomly fails.


def make_token_batches(chunks, enc, max_batch_tokens: int = 7000):
    # Group chunks into batches so the total tokens per API call stays under the limit.
    # 7000 leaves some buffer under the 8192 token cap.

    batches = []
    current = []
    current_tokens = 0

    for c in chunks:
        text = c.get("text", "")
        if not isinstance(text, str):
            text = str(text)

        text = text.strip()
        if not text:
            continue

        n = len(enc.encode(text))

        # If a single chunk is huge, truncate it so it can still be embedded.
        if n > max_batch_tokens:
            text = enc.decode(enc.encode(text)[:max_batch_tokens])
            c = {**c, "text": text}
            n = len(enc.encode(text))

        # Start a new batch if adding this chunk would exceed our budget.
        if current and current_tokens + n > max_batch_tokens:
            batches.append(current)
            current = []
            current_tokens = 0

        current.append(c)
        current_tokens += n

    if current:
        batches.append(current)

    return batches


def embed_texts(client: OpenAI, texts):
    # Embed a list of strings and return a (n, dim) float32 NumPy array.
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = [item.embedding for item in resp.data]
    return np.array(vecs, dtype=np.float32)

enc = tiktoken.get_encoding("cl100k_base")

def build_vector_index(client: OpenAI, chunks, enc, max_batch_tokens: int = 7000):
    # Build an in-memory vector index.
    # X[i] matches metas[i] (same order).

    all_vecs = []
    metas = []

    batches = make_token_batches(chunks, enc, max_batch_tokens=max_batch_tokens)
    # Pack chunks into token-safe batches so we don’t exceed the API limit.

    for batch in batches:
        batch_texts = [c["text"] for c in batch]
        vecs = embed_texts(client, batch_texts)
        all_vecs.append(vecs)
        metas.extend(batch)

    X = np.vstack(all_vecs) if all_vecs else np.zeros((0, 1536), dtype=np.float32)
    return X, metas

X, metas = build_vector_index(client, chunks, enc)
print("Embeddings shape:", X.shape)


Embeddings shape: (1112, 1536)


In [6]:
def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    cosine similarity between rows of a and rows of b.
    """
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
    return np.matmul(a_norm, b_norm.T)


def retrieve_relevant_chunks(client: OpenAI,
                             X: np.ndarray,
                             metas: List[Dict],
                             question: str,
                             top_k: int = TOP_K) -> List[Dict]:
    """
    Embed question, return top_k most similar chunks.
    """
    q_vec = embed_texts(client, [question])  # (1, dim)
    sims = cosine_sim_matrix(q_vec, X)[0]    # (num_chunks,)
    idxs = np.argsort(-sims)[:top_k]

    results = []
    for rank, i in enumerate(idxs):
        m = metas[i]
        results.append({
            "rank": rank,
            "score": float(sims[i]),
            "source": m["source"],
            "chunk_id": m["chunk_id"],
            "text": m["text"]
        })
    return results


def build_system_prompt() -> str:
    return (
        "You are a helpful assistant. "
        "Answer using ONLY the provided context chunks and the chat history. "
        "If you are not sure, say you are not sure. "
        "Cite the source like (source.txt #3) when relevant."
    )


def build_user_prompt(question: str,
                      history: List[Dict],
                      retrieved_chunks: List[Dict]) -> str:
    """
    Build the message we send to the model:
    - short chat history
    - retrieved context
    - current question
    """
    # include last N turns of conversation
    recent_history = history[-MAX_CONTEXT_MESSAGES:]

    hist_lines = []
    if recent_history:
        for turn in recent_history:
            hist_lines.append(f"User: {turn['user']}\nAssistant: {turn['assistant']}\n")
        hist_block = "\n".join([f"User: {t['user']}\nAssistant: {t['assistant']}" for t in history[-4:]])
    else:
        hist_block = "No previous history in this chat.\n"

    ctx_lines = []
    for c in retrieved_chunks:
        ctx_lines.append(
            f"[{c['source']} #{c['chunk_id']} score={c['score']:.3f}]\n{c['text']}\n"
        )
    ctx_block = "\n".join(ctx_lines)

    user_block = (
        "CHAT HISTORY:\n"
        f"{hist_block}\n"
        "RELEVANT DOCUMENT CONTEXT:\n"
        f"{ctx_block}\n"
        "QUESTION:\n"
        f"{question}\n"
        "INSTRUCTIONS:\n"
        "Use the context to answer. If context does not contain the answer, say you don't know.\n"
        "When you reference info from context, include (filename #chunk_id).\n"
    )

    return user_block


def ask_llm(client: OpenAI,
            system_prompt: str,
            user_prompt: str) -> str:
    """
    Call Responses API.
    """
    resp = client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
    )

    out_txt = ""
    for out_item in resp.output:
        for content_part in out_item.content:
            if content_part.type == "output_text":
                out_txt += content_part.text
    return out_txt.strip()

def rewrite_question_with_history(client, history, question: str) -> str:
    # Rewrite follow-up questions so they make sense on their own (fixes "it/that/they" problems).
    if not history:
        return question

    last_turns = history[-2:]  # keep this small and stable
    history_text = "\n".join(
        [f"User: {t['user']}\nAssistant: {t['assistant']}" for t in last_turns]
    )

    prompt = (
        "Rewrite the user's question so it is fully self-contained.\n"
        "If the question already makes sense alone, keep it the same.\n"
        "Do not answer the question. Only rewrite it.\n\n"
        f"Conversation:\n{history_text}\n\n"
        f"User question: {question}\n"
        "Rewritten question:"
    )

    resp = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return resp.output_text.strip()


In [7]:
class ChatSession:
    """
    One ChatSession = one 'run'.
    It has:
    - its own conversation_history (RAM only)
    - access to shared X/metas
    When you make a NEW ChatSession(), it's a fresh memory.
    """

    def __init__(self, client, X, metas):
        # Save the OpenAI client and the pre-built vector index so we can retrieve context on each question.
        self.client = client
        self.X = X
        self.metas = metas
        self.history: List[Dict] = []  # wiped every new ChatSession
        self.system_prompt = build_system_prompt()

    def ask(self, user_question: str) -> str:
        standalone_question = rewrite_question_with_history(
            self.client, self.history, user_question
        )
        # Now retrieval sees the real topic, not a vague "it".

        retrieved = retrieve_relevant_chunks(
            self.client, self.X, self.metas, standalone_question, TOP_K
        )

        user_prompt = build_user_prompt(
            question=standalone_question,
            history=self.history,
            retrieved_chunks=retrieved
        )

        answer = ask_llm(self.client, self.system_prompt, user_prompt)

        self.history.append({"user": user_question, "assistant": answer})
        return answer

# Create a fresh chat session
chat = ChatSession(client, X, metas)
print("Chat ready. New memory started.")


Chat ready. New memory started.


In [8]:
# Notebook-only helper to display chat messages nicely using Markdown formatting.

from IPython.display import display, Markdown

def pretty_chat(role, text):
    if role == "user":
        display(Markdown(f"**🧑‍💻 You:** {text}"))
    else:
        display(Markdown(f"**🤖 Assistant:** {text}"))


In [17]:
# Ask one question through the chat session and display the user message + model answer in notebook-friendly format.

q = "What is machine learning? What are the main types of machine learning?"
pretty_chat("user", q)
a = chat.ask(q)
pretty_chat("assistant", a)

**🧑‍💻 You:** What is machine learning? What are the main types of machine learning?

**🤖 Assistant:** Machine Learning is the science (and art) of programming computers so they can learn from data. It is defined as the field of study that gives computers the ability to learn without being explicitly programmed. A more engineering-oriented definition states that a computer program learns from experience E with respect to some task T and some performance measure P if its performance on T, as measured by P, improves with experience E (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #31).

The main types of machine learning systems can be classified as follows:

1. **Supervised Learning**: The training set includes desired solutions, called labels. An example is a spam filter trained with emails labeled as spam or ham (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #37).

2. **Unsupervised Learning**: The system learns from data without labeled responses, including tasks like clustering and anomaly detection (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #36).

3. **Semisupervised Learning**: A combination of supervised and unsupervised learning, using both labeled and unlabeled data.

4. **Reinforcement Learning**: The learning system, called an agent, learns by observing the environment, selecting actions, and receiving rewards or penalties (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #43).

In [18]:
# Ask a question in a fresh session and print the answer (simple, no notebook formatting).

chat2 = ChatSession(client, X, metas)
chat2.ask("What is overfitting?")

'Overfitting is when a machine learning model performs well on the training data but does not generalize well to new, unseen data. This occurs when the model is too complex relative to the amount and noisiness of the training data, leading it to detect patterns in the noise rather than in the actual underlying data. As a result, the model may make inaccurate predictions on new instances (source.txt #57).'

In [ ]:
# Interactive chat loop that keeps session history, so follow-up questions have context until you exit.

while True:
    q = input("🧑‍💻 You: ").strip()
    if q.lower() in ["exit", "quit"]:
        print("👋 Bye!")
        break
    
    pretty_chat("user", q)
    a = chat.ask(q)
    pretty_chat("assistant", a)
    print()  # adds a blank line between turns


**🧑‍💻 You:** What is a loss function?

**🤖 Assistant:** A loss function in the context of machine learning is a method used to measure how well a model's predictions match the actual data. It quantifies the difference between the predicted values and the true values, allowing the model to adjust its parameters to minimize this difference during training. For example, the mean squared error and Huber loss are types of loss functions used to evaluate the performance of regression models (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506).

**🧑‍💻 You:** What are the most popular types of them?

**🤖 Assistant:** The most popular types of loss functions used in machine learning include:

1. **Mean Squared Error (MSE)**: Commonly used for regression tasks, it measures the average of the squares of the errors between predicted and actual values.

2. **Mean Absolute Error (MAE)**: Another loss function for regression, it measures the average of the absolute differences between predicted and actual values.

3. **Huber Loss**: A combination of MSE and MAE, it is less sensitive to outliers in data than MSE.

4. **Cross-Entropy Loss**: Used for classification tasks, it measures the performance of a model whose output is a probability value between 0 and 1 (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506). 

These loss functions help in evaluating the performance of models and guiding the training process.

**🧑‍💻 You:** What do I need to know to learn them?

**🤖 Assistant:** To learn about the most popular types of loss functions in machine learning, you should focus on the following foundational knowledge and skills:

1. **Understanding Loss Functions**: Grasp the concept of loss functions as methods to measure how well a model's predictions match actual data. This includes knowing how they quantify the difference between predicted and true values (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506).

2. **Types of Loss Functions**: Familiarize yourself with the most popular loss functions:
   - **Mean Squared Error (MSE)**: Used for regression tasks, measuring the average of the squares of errors.
   - **Mean Absolute Error (MAE)**: Another regression loss function that measures the average of absolute differences.
   - **Huber Loss**: A combination of MSE and MAE, less sensitive to outliers.
   - **Cross-Entropy Loss**: Used for classification tasks, measuring the performance of models outputting probabilities (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506).

3. **Application of Loss Functions**: Learn how to select appropriate loss functions based on the type of problem you are solving, such as regression or classification, and understand the implications of using different loss functions on model performance.

4. **Custom Loss Functions**: Gain skills in creating custom loss functions when standard ones do not meet the needs of your specific dataset or problem (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #510).

5. **Evaluation of Model Performance**: Understand how loss functions guide the training process and help evaluate the performance of models, which is crucial for fine-tuning and improving model accuracy.

These foundational skills will enable you to effectively work with loss functions in machine learning.

**🧑‍💻 You:** 

**🤖 Assistant:** To learn about the most popular types of loss functions in machine learning, you should focus on the following foundational knowledge and skills:

1. **Understanding Loss Functions**: Grasp the concept of loss functions as methods to measure how well a model's predictions match actual data. This includes knowing how they quantify the difference between predicted and true values (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506).

2. **Types of Loss Functions**: Familiarize yourself with the most popular loss functions:
   - **Mean Squared Error (MSE)**: Used for regression tasks, measuring the average of the squares of errors.
   - **Mean Absolute Error (MAE)**: Another regression loss function that measures the average of absolute differences.
   - **Huber Loss**: A combination of MSE and MAE, less sensitive to outliers.
   - **Cross-Entropy Loss**: Used for classification tasks, measuring the performance of models outputting probabilities (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #506).

3. **Application of Loss Functions**: Learn how to select appropriate loss functions based on the type of problem you are solving, such as regression or classification, and understand the implications of using different loss functions on model performance.

4. **Custom Loss Functions**: Gain skills in creating custom loss functions when standard ones do not meet the needs of your specific dataset or problem (Aurélien Géron - Hands-On Machine Learning with Scikit Learn, Keras.pdf #510).

5. **Evaluation of Model Performance**: Understand how loss functions guide the training process and help evaluate the performance of models, which is crucial for fine-tuning and improving model accuracy.

These foundational skills will enable you to effectively work with loss functions in machine learning.